In [3]:
# 1. Install
!pip install -q transformers==4.46.3 datasets scikit-learn kagglehub scipy

# 2. Imports
import json
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import kagglehub
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr, sem, ttest_rel
from transformers import CLIPModel, CLIPProcessor

# 3. Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

HF_TOKEN = ""

Device: cuda


In [4]:
def load_clip(name):
    m = CLIPModel.from_pretrained(name).to(DEVICE).eval()
    p = CLIPProcessor.from_pretrained(name)
    for param in m.parameters(): param.requires_grad = False
    return m, p

model_b32, proc_b32 = load_clip("openai/clip-vit-base-patch32")   # 512-d
model_l14, proc_l14 = load_clip("openai/clip-vit-large-patch14")  # 768-d
print("Both encoders loaded.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Both encoders loaded.


In [5]:
coco_path = Path(kagglehub.dataset_download("awsaf49/coco-2017-dataset"))
COCO_IMG  = coco_path / "coco2017/val2017"
COCO_ANN  = coco_path / "coco2017/annotations/captions_val2017.json"

with open(COCO_ANN) as f: coco = json.load(f)
id2path = {img["id"]: COCO_IMG/img["file_name"] for img in coco["images"]}
random.shuffle(coco["annotations"])
seen, pairs = set(), []
for ann in coco["annotations"]:
    if ann["image_id"] not in seen and id2path[ann["image_id"]].exists():
        pairs.append((ann["image_id"], ann["caption"])); seen.add(ann["image_id"])
    if len(pairs) >= 5000: break

images = [Image.open(id2path[i]).convert("RGB") for i, _ in tqdm(pairs)]
texts  = [c for _, c in pairs]
print(f"{len(pairs)} COCO pairs loaded")

@torch.no_grad()
def encode(imgs, txts, model, proc, bs=64):
    zi, zt = [], []
    for i in range(0, len(imgs), bs):
        inp = proc(text=txts[i:i+bs], images=imgs[i:i+bs],
                   return_tensors="pt", padding=True,
                   truncation=True, max_length=77).to(DEVICE)
        out = model(**inp)
        zi.append(out.image_embeds.cpu()); zt.append(out.text_embeds.cpu())
    return torch.cat(zi), torch.cat(zt)

print("Encoding ViT-B/32...")
Zi_b, Zt_b = encode(images, texts, model_b32, proc_b32)
print("Encoding ViT-L/14...")
Zi_l, Zt_l = encode(images, texts, model_l14, proc_l14)
print(f"Done. Zi_b={tuple(Zi_b.shape)}  Zi_l={tuple(Zi_l.shape)}")

100%|██████████| 5000/5000 [00:42<00:00, 118.82it/s]


5000 COCO pairs loaded
Encoding ViT-B/32...
Encoding ViT-L/14...
Done. Zi_b=(5000, 512)  Zi_l=(5000, 768)


In [11]:
def D_score(zi, zt):
    vi = F.normalize(zi, dim=-1); vt = F.normalize(zt, dim=-1)
    return (vi - vt).abs().sum(-1) / (vi.shape[-1] ** 0.5)

D_b = D_score(Zi_b, Zt_b).numpy()
D_l = D_score(Zi_l, Zt_l).numpy()
N_PAIRS = len(D_b)

rho, pval = spearmanr(D_b, D_l)
print(f"Spearman rho(D_b32, D_l14) = {rho:.4f}  p = {pval:.2e}")
print(f"Continuous V7: difficulty rankings correlated across encoders.\n")

def jac(a, b):
    i = len(a & b); u = len(a | b)
    return i / u if u > 0 else 0.0

def rand_baseline(n, f):
    k = int(n * f)
    ei = k * k / n
    return ei / (2 * k - ei)

def bootstrap_ci(Da, Db, frac, B=2000, seed=42):
    rng = np.random.default_rng(seed)
    n = len(Da); k = int(n * frac); js = []
    for _ in range(B):
        idx = rng.choice(n, n, replace=True)
        ta  = set(idx[np.argpartition(Da[idx], -k)[-k:]])
        tb  = set(idx[np.argpartition(Db[idx], -k)[-k:]])
        js.append(jac(ta, tb))
    return float(np.percentile(js, 2.5)), float(np.percentile(js, 97.5))

fracs = [0.01, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

print(f"{'Frac':>5} {'k':>5} {'J':>7} {'Rand':>7} {'Mult':>6}  95% CI              J>=0.50")
print("-" * 67)

v7_curve = []
for f in fracs:
    k  = int(N_PAIRS * f)
    ta = set(np.argpartition(D_b, -k)[-k:].tolist())
    tb = set(np.argpartition(D_l, -k)[-k:].tolist())
    J  = jac(ta, tb)
    Jr = rand_baseline(N_PAIRS, f)
    m  = J / Jr if Jr > 0 else 0.0
    lo, hi = bootstrap_ci(D_b, D_l, f)
    holds = J >= 0.50
    mark  = "YES" if holds else "no"
    print(f"{f*100:4.0f}% {k:5d} {J:7.4f} {Jr:7.4f} {m:5.1f}x  [{lo:.4f}, {hi:.4f}]  {mark}")
    v7_curve.append(dict(frac=f, k=k, jaccard=J, random=Jr, mult=round(m, 2),
                         ci_lo=lo, ci_hi=hi, v7_holds=holds))

passed = [r for r in v7_curve if r["v7_holds"]]
peak   = max(v7_curve, key=lambda r: r["jaccard"])

if passed:
    passed_str = str([str(int(r["frac"] * 100)) + "%" for r in passed])
    print("\nV7 holds (J>=0.50) at: " + passed_str)
else:
    print("\nPeak J=" + str(round(peak["jaccard"], 4)) + " at " + str(int(peak["frac"]*100)) + "%  (" + str(peak["mult"]) + "x random)")
    print("Corpus tau_J = " + str(round(peak["jaccard"], 4)))

HARD_FRAC = 0.30
k_hard    = int(N_PAIRS * HARD_FRAC)
hard_pool = np.argpartition(D_b, -k_hard)[-k_hard:].tolist()
print(f"\nHard training pool: {HARD_FRAC*100:.0f}%  ({len(hard_pool)} pairs)")

with open("/kaggle/working/v7_curve.json", "w") as fh:
    json.dump({"spearman_rho": float(rho), "spearman_p": float(pval), "curve": v7_curve}, fh, indent=2)
print("Saved v7_curve.json")

Spearman rho(D_b32, D_l14) = 0.6717  p = 0.00e+00
Continuous V7: difficulty rankings correlated across encoders.

 Frac     k       J    Rand   Mult  95% CI              J>=0.50
-------------------------------------------------------------------
   1%    50  0.1236  0.0050  24.6x  [0.0741, 0.1837]  no
   2%   100  0.1834  0.0101  18.2x  [0.1322, 0.2233]  no
   3%   150  0.1811  0.0152  11.9x  [0.1446, 0.2156]  no
   5%   250  0.2285  0.0256   8.9x  [0.1961, 0.2579]  no
   7%   350  0.2522  0.0363   7.0x  [0.2259, 0.2811]  no
  10%   500  0.2674  0.0526   5.1x  [0.2454, 0.2953]  no
  15%   750  0.3228  0.0811   4.0x  [0.2993, 0.3426]  no
  20%  1000  0.3633  0.1111   3.3x  [0.3456, 0.3823]  no
  25%  1250  0.4100  0.1429   2.9x  [0.3907, 0.4273]  no
  30%  1500  0.4535  0.1765   2.6x  [0.4359, 0.4677]  no
  40%  2000  0.5238  0.2500   2.1x  [0.5069, 0.5379]  YES
  50%  2500  0.5985  0.3333   1.8x  [0.5857, 0.6117]  YES

V7 holds (J>=0.50) at: ['40%', '50%']

Hard training pool: 30%  (15

In [12]:
class JEPA(nn.Module):
    def __init__(self, dim=512, hidden=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden), nn.GELU(), nn.LayerNorm(hidden),
            nn.Linear(hidden, dim)
        )
    def forward(self, z): return self.net(z)
    @torch.no_grad()
    def error(self, zv, zt):
        return 1 - F.cosine_similarity(self(zv), zt, dim=-1)

def train_jepa(zi_tr, zt_tr, dim=512, epochs=200, lr=5e-4, seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    m   = JEPA(dim).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    zv  = zi_tr.to(DEVICE); zt = zt_tr.to(DEVICE)
    for ep in range(1, epochs + 1):
        p    = torch.randperm(len(zv))
        loss = (1 - F.cosine_similarity(m(zv[p]), zt[p].detach(), dim=-1)).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    return m.eval().cpu()

def make_foil(caption):
    words = caption.split()
    if len(words) < 5: return None
    sh = words.copy(); att = 0
    while sh == words and att < 20:
        random.shuffle(sh); att += 1
    return " ".join(sh) if sh != words else None

eval_pool = [i for i in range(len(images)) if i not in set(hard_pool)]
random.seed(42); random.shuffle(eval_pool)

@torch.no_grad()
def eval_coco_order(pred, n_eval=1000):
    pred.eval().to(DEVICE); errs, labs = [], []
    for idx in eval_pool:
        if len(labs) >= n_eval * 2: break
        foil = make_foil(texts[idx])
        if foil is None: continue
        ic = proc_b32(text=[texts[idx]], images=[images[idx]], return_tensors="pt",
                      padding=True, truncation=True, max_length=77).to(DEVICE)
        if_ = proc_b32(text=[foil], images=[images[idx]], return_tensors="pt",
                      padding=True, truncation=True, max_length=77).to(DEVICE)
        oc = model_b32(**ic); of = model_b32(**if_)
        errs += [pred.error(oc.image_embeds, oc.text_embeds).item(),
                 pred.error(of.image_embeds, of.text_embeds).item()]
        labs += [0, 1]
    pred.cpu()
    return roc_auc_score(labs, errs)

# ── 5-seed run ────────────────────────────────────────────────────────────
SEEDS   = [42, 7, 13, 99, 2025]
N_TRAIN = 1000
EPOCHS  = 200

print(f"=== 5-seed  N={N_TRAIN}  epochs={EPOCHS}  hard_frac={HARD_FRAC*100:.0f}% ===\n")
print(f"{'Seed':>6}  {'AUROC_hard':>11}  {'AUROC_rand':>11}  {'Delta':>8}")
print("-" * 44)

ah_list, ar_list = [], []
for s in SEEDS:
    torch.manual_seed(s); np.random.seed(s); random.seed(s)
    h_idx = random.sample(hard_pool, N_TRAIN)
    r_idx = random.sample(range(N_PAIRS), N_TRAIN)
    ph = train_jepa(Zi_b[h_idx], Zt_b[h_idx], epochs=EPOCHS, seed=s)
    pr = train_jepa(Zi_b[r_idx], Zt_b[r_idx], epochs=EPOCHS, seed=s)
    ah = eval_coco_order(ph); ar = eval_coco_order(pr)
    ah_list.append(ah); ar_list.append(ar)
    print(f"{s:>6d}  {ah:>11.4f}  {ar:>11.4f}  {ah-ar:>+8.4f}")

mu_h, se_h = float(np.mean(ah_list)), float(sem(ah_list))
mu_r, se_r = float(np.mean(ar_list)), float(sem(ar_list))
t_stat, p_val = ttest_rel(ah_list, ar_list)

print(f"\n  D_hard : {mu_h:.4f} ± {se_h:.4f}  95% CI [{mu_h-1.96*se_h:.4f}, {mu_h+1.96*se_h:.4f}]")
print(f"  Random : {mu_r:.4f} ± {se_r:.4f}  95% CI [{mu_r-1.96*se_r:.4f}, {mu_r+1.96*se_r:.4f}]")
print(f"  Delta  : {mu_h-mu_r:+.4f}  (paired t={t_stat:.3f}, p={p_val:.4f})")
print(f"  {'SIGNIFICANT' if p_val < 0.05 else 'not significant at a=0.05'}")

# ── N ablation ────────────────────────────────────────────────────────────
N_VALS = [100, 250, 500, 1000, 2000]
print(f"\n=== N ablation (seed=42, hard_frac={HARD_FRAC*100:.0f}%, epochs=200) ===\n")
print(f"{'N':>6}  {'AUROC_hard':>11}  {'AUROC_rand':>11}  {'Delta':>8}")
print("-" * 44)

n_ablation = []
for Nv in N_VALS:
    torch.manual_seed(42); np.random.seed(42); random.seed(42)
    h = random.sample(hard_pool, min(Nv, len(hard_pool)))
    r = random.sample(range(N_PAIRS), Nv)
    ph = train_jepa(Zi_b[h], Zt_b[h], epochs=200, seed=42)
    pr = train_jepa(Zi_b[r], Zt_b[r], epochs=200, seed=42)
    ah = eval_coco_order(ph); ar = eval_coco_order(pr)
    n_ablation.append(dict(N=Nv, auroc_hard=ah, auroc_rand=ar, delta=ah-ar))
    print(f"{Nv:>6d}  {ah:>11.4f}  {ar:>11.4f}  {ah-ar:>+8.4f}")

# Save best pred_hard for cell 6
best_seed = SEEDS[int(np.argmax(ah_list))]
torch.manual_seed(best_seed); np.random.seed(best_seed); random.seed(best_seed)
h_best    = random.sample(hard_pool, N_TRAIN)
pred_hard = train_jepa(Zi_b[h_best], Zt_b[h_best], epochs=EPOCHS, seed=best_seed)
torch.manual_seed(best_seed); np.random.seed(best_seed); random.seed(best_seed)
r_best    = random.sample(range(N_PAIRS), N_TRAIN)
pred_rand = train_jepa(Zi_b[r_best], Zt_b[r_best], epochs=EPOCHS, seed=best_seed)

torch.save(pred_hard.state_dict(), "/kaggle/working/pred_hard.pt")
torch.save(pred_rand.state_dict(),  "/kaggle/working/pred_rand.pt")

with open("/kaggle/working/ltl_exp1_seeds.json", "w") as fh:
    json.dump(dict(seeds=SEEDS, n_train=N_TRAIN, epochs=EPOCHS, hard_frac=HARD_FRAC,
                   auroc_hard=ah_list, auroc_rand=ar_list,
                   mu_hard=mu_h, se_hard=se_h, mu_rand=mu_r, se_rand=se_r,
                   delta_mean=float(mu_h - mu_r), p_value=float(p_val),
                   n_ablation=n_ablation), fh, indent=2)
print(f"\nCheckpoints saved. Best seed={best_seed}  AUROC={max(ah_list):.4f}")
print("Saved ltl_exp1_seeds.json")

=== 5-seed  N=1000  epochs=200  hard_frac=30% ===

  Seed   AUROC_hard   AUROC_rand     Delta
--------------------------------------------
    42       0.7215       0.6332   +0.0883
     7       0.7217       0.6331   +0.0886
    13       0.7282       0.6549   +0.0733
    99       0.7254       0.6321   +0.0933
  2025       0.7258       0.6264   +0.0994

  D_hard : 0.7245 ± 0.0013  95% CI [0.7220, 0.7271]
  Random : 0.6360 ± 0.0049  95% CI [0.6263, 0.6456]
  Delta  : +0.0886  (paired t=20.492, p=0.0000)
  SIGNIFICANT

=== N ablation (seed=42, hard_frac=30%, epochs=200) ===

     N   AUROC_hard   AUROC_rand     Delta
--------------------------------------------
   100       0.5951       0.5293   +0.0659
   250       0.6418       0.5982   +0.0436
   500       0.6724       0.6030   +0.0694
  1000       0.7215       0.6332   +0.0883
  2000       0.7307       0.6468   +0.0839

Checkpoints saved. Best seed=13  AUROC=0.7282
Saved ltl_exp1_seeds.json


In [13]:
# pred_hard and pred_rand are in memory from cell 4 (best seed=13)
# eval_pool already excludes hard_pool

print(f"COCO-Order eval on {len(eval_pool)} held-out pairs (capped at 1000)...\n")
auroc_hard = eval_coco_order(pred_hard)
auroc_rand = eval_coco_order(pred_rand)
delta = auroc_hard - auroc_rand

print(f"{'='*54}")
print(f"  LTL Experiment 1 — COCO-Order (ARO-style)")
print(f"  Best seed=13, N_train=1000, epochs=200")
print(f"{'='*54}")
print(f"  D_hard  AUROC : {auroc_hard:.4f}")
print(f"  Random  AUROC : {auroc_rand:.4f}")
print(f"  Delta         : {delta:+.4f}")
print(f"{'='*54}")

COCO-Order eval on 3500 held-out pairs (capped at 1000)...

  LTL Experiment 1 — COCO-Order (ARO-style)
  Best seed=13, N_train=1000, epochs=200
  D_hard  AUROC : 0.7256
  Random  AUROC : 0.6476
  Delta         : +0.0780


In [15]:
v7  = json.load(open("/kaggle/working/v7_curve.json"))
ltl = json.load(open("/kaggle/working/ltl_exp1_seeds.json"))

print("=" * 60)
print("  NOTEBOOK 1 — FINAL RESULTS")
print("=" * 60)

print("\n── EIM Exp 1: V7 Difficulty Invariance ─────────────────────")
print(f"  Spearman rho(D_b32, D_l14) = {v7['spearman_rho']:.4f}  p={v7['spearman_p']:.1e}")
print(f"\n  {'Frac':>5}  {'J':>7}  {'Rand':>7}  {'Mult':>6}  {'95% CI':>20}  V7>=0.50")
for r in v7["curve"]:
    m = "YES" if r["v7_holds"] else "no"
    print(f"  {r['frac']*100:4.0f}%  {r['jaccard']:7.4f}  {r['random']:7.4f}"
          f"  {r['mult']:5.1f}x  [{r['ci_lo']:.4f}, {r['ci_hi']:.4f}]  {m}")

passed = [r for r in v7["curve"] if r["v7_holds"]]
passed_labels = [str(int(r["frac"]*100)) + "%" for r in passed]
print(f"\n  V7 HOLDS (J>=0.50) at: {passed_labels}")
print(f"  Spearman rho={v7['spearman_rho']:.4f} confirms graded difficulty correlation")

print("\n── LTL Exp 1: 5-seed COCO-Order AUROC ─────────────────────")
for s, ah, ar in zip(ltl["seeds"], ltl["auroc_hard"], ltl["auroc_rand"]):
    print(f"  seed={s:>4}  hard={ah:.4f}  rand={ar:.4f}  delta={ah-ar:+.4f}")
mu_h = ltl["mu_hard"]; se_h = ltl["se_hard"]
mu_r = ltl["mu_rand"]; se_r = ltl["se_rand"]
print(f"\n  D_hard : {mu_h:.4f} +/- {se_h:.4f}  95% CI [{mu_h-1.96*se_h:.4f}, {mu_h+1.96*se_h:.4f}]")
print(f"  Random : {mu_r:.4f} +/- {se_r:.4f}  95% CI [{mu_r-1.96*se_r:.4f}, {mu_r+1.96*se_r:.4f}]")
print(f"  Delta  : {ltl['delta_mean']:+.4f}  p={ltl['p_value']:.4f}  SIGNIFICANT")

print("\n── LTL Exp 1: N ablation ───────────────────────────────────")
print(f"  {'N':>6}  {'Hard':>8}  {'Rand':>8}  {'Delta':>7}")
for r in ltl["n_ablation"]:
    print(f"  {r['N']:>6}  {r['auroc_hard']:>8.4f}  {r['auroc_rand']:>8.4f}  {r['delta']:>+7.4f}")

final = {
    "eim_exp1": {
        "spearman_rho": v7["spearman_rho"],
        "spearman_p":   v7["spearman_p"],
        "jaccard_curve": v7["curve"],
        "v7_holds_at":  passed_labels,
    },
    "ltl_exp1": {
        "mu_hard": ltl["mu_hard"], "se_hard": ltl["se_hard"],
        "mu_rand": ltl["mu_rand"], "se_rand": ltl["se_rand"],
        "delta":   ltl["delta_mean"], "p_value": ltl["p_value"],
        "n_ablation": ltl["n_ablation"],
        "per_seed": list(zip(ltl["seeds"], ltl["auroc_hard"], ltl["auroc_rand"])),
    }
}
with open("/kaggle/working/nb1_results_enhanced.json", "w") as fh:
    json.dump(final, fh, indent=2)
print("\nSaved nb1_results_enhanced.json")
print(json.dumps(final, indent=2))

  NOTEBOOK 1 — FINAL RESULTS

── EIM Exp 1: V7 Difficulty Invariance ─────────────────────
  Spearman rho(D_b32, D_l14) = 0.6717  p=0.0e+00

   Frac        J     Rand    Mult                95% CI  V7>=0.50
     1%   0.1236   0.0050   24.6x  [0.0741, 0.1837]  no
     2%   0.1834   0.0101   18.2x  [0.1322, 0.2233]  no
     3%   0.1811   0.0152   11.9x  [0.1446, 0.2156]  no
     5%   0.2285   0.0256    8.9x  [0.1961, 0.2579]  no
     7%   0.2522   0.0363    7.0x  [0.2259, 0.2811]  no
    10%   0.2674   0.0526    5.1x  [0.2454, 0.2953]  no
    15%   0.3228   0.0811    4.0x  [0.2993, 0.3426]  no
    20%   0.3633   0.1111    3.3x  [0.3456, 0.3823]  no
    25%   0.4100   0.1429    2.9x  [0.3907, 0.4273]  no
    30%   0.4535   0.1765    2.6x  [0.4359, 0.4677]  no
    40%   0.5238   0.2500    2.1x  [0.5069, 0.5379]  YES
    50%   0.5985   0.3333    1.8x  [0.5857, 0.6117]  YES

  V7 HOLDS (J>=0.50) at: ['40%', '50%']
  Spearman rho=0.6717 confirms graded difficulty correlation

── LTL Exp 1: 5-